In [ ]:
import time
import os
import mlflow
import urllib3
import requests
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from requests.auth import HTTPBasicAuth
from threading import Lock
from collections import deque
from datetime import datetime, timedelta

# Disable warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

print('Libraries imported successfully')


In [ ]:
class ProductionSafeAdaptiveBounds:
    '\n    Production-ready adaptive bounds manager with safety validation.\n    Maintains adaptive learning while enforcing safety constraints.\n    '
    
    def __init__(self, buffer_margin=0.12):  # Reduced from 0.15 for safety
        self.buffer_margin = buffer_margin
        self.data_buffer_size = 180  # 3-minute window
        self.update_interval = timedelta(minutes=1)
        self.min_data_points = 30    # Need 30 seconds minimum
        
        # Rolling data buffers
        self.recent_data = {i: deque(maxlen=self.data_buffer_size) for i in range(14)}
        
        # PRODUCTION SAFETY: Emergency minimums (non-negotiable)
        self.emergency_safety_limits = {
            1: 150.0,   # Total air flow minimum
            2: 30.0,    # Wind box DP minimum
            6: 120.0,   # Steam flow minimum
            9: 15.45,   # Coal flow minimum
            10: 190.0,  # APH outlet temp minimum
            'air_fuel_ratio_min': 2.0  # Absolute minimum air/fuel ratio
        }
        
        # Start with engineering design bounds (not wide open bounds)
        self.current_bounds = [
            (2.1, 2.8), (150, 300), (30, 100), (10, 50), (10, 50), (0.5, 1.5),
            (120, 300), (380, 420), (290, 350), (15.45, 90), (240, 300),
            (150, 250), (110, 170), (2, 6)
        ]
        
        self.current_constraint_mins = {
            'total_air_flow_min': 150.0,
            'wind_box_dp_min': 30.0,
            'steam_flow_min': 120.0,
            'coal_flow_min': 15.45,
            'air_fuel_ratio_min': 2.11,
            'aph_outlet_min': 190.0
        }
        
        self.last_update = datetime.now()
        print('Production-safe adaptive bounds manager initialized')

    def add_real_time_data(self, parameter_values):
        'Add data with quality validation'
        for i, value in enumerate(parameter_values):
            if value is not None and isinstance(value, (int, float)) and not np.isnan(value):
                # Additional validation: reject extreme outliers
                if i in self.emergency_safety_limits:
                    safety_min = self.emergency_safety_limits[i]
                    if value >= safety_min * 0.5:  # Reject values below 50% of safety minimum
                        self.recent_data[i].append(value)
                else:
                    self.recent_data[i].append(value)

    def calculate_adaptive_bounds_and_constraints(self):
        'Calculate adaptive bounds with safety enforcement'
        new_bounds = []
        constraint_mins = {}
        
        for i in range(14):
            data = list(self.recent_data[i])
            
            if len(data) > self.min_data_points:
                # Use 10th/90th percentiles for production stability (not 5th/95th)
                p10 = np.percentile(data, 10)
                p90 = np.percentile(data, 90)
                
                # Apply buffer margin
                range_buffer = (p90 - p10) * self.buffer_margin
                new_min = max(0, p10 - range_buffer)
                new_max = p90 + range_buffer
                
                # CRITICAL: Apply emergency safety overrides
                if i in self.emergency_safety_limits:
                    new_min = max(new_min, self.emergency_safety_limits[i])
                
                # Limit change rate to 3% per minute for production stability
                current_min, current_max = self.current_bounds[i]
                max_change_rate = 0.03
                
                new_min = max(current_min * (1 - max_change_rate), 
                             min(current_min * (1 + max_change_rate), new_min))
                new_max = max(current_max * (1 - max_change_rate), 
                             min(current_max * (1 + max_change_rate), new_max))
                
                new_bounds.append((new_min, new_max))
                
                # Set constraint minimums for critical parameters
                if i in [1, 2, 6, 9, 10]:
                    # Use 20th percentile for more conservative constraints
                    learned_min = np.percentile(data, 20)
                    safety_min = self.emergency_safety_limits.get(i, 0)
                    constraint_mins[i] = max(learned_min, safety_min)
            else:
                # Keep current bounds if insufficient data
                new_bounds.append(self.current_bounds[i])
        
        # Map constraints
        constraint_map = {
            1: 'total_air_flow_min',
            2: 'wind_box_dp_min', 
            6: 'steam_flow_min',
            9: 'coal_flow_min',
            10: 'aph_outlet_min'
        }
        
        new_constraint_mins = {constraint_map[i]: constraint_mins[i] 
                             for i in constraint_mins if i in constraint_map}
        
        # Calculate air/fuel ratio with safety floor
        air_data = list(self.recent_data[1])
        coal_data = list(self.recent_data[9])
        
        if len(air_data) > self.min_data_points and len(coal_data) > self.min_data_points:
            ratios = [a/c for a, c in zip(air_data, coal_data) if c > 0]
            if ratios:
                learned_ratio = np.percentile(ratios, 20)  # Conservative 20th percentile
                new_constraint_mins['air_fuel_ratio_min'] = max(learned_ratio, 
                                                              self.emergency_safety_limits['air_fuel_ratio_min'])
            else:
                new_constraint_mins['air_fuel_ratio_min'] = 2.11
        else:
            new_constraint_mins['air_fuel_ratio_min'] = 2.11
            
        return new_bounds, new_constraint_mins

    def validate_bounds_compatibility(self, new_bounds, new_constraint_mins):
        'PRODUCTION CRITICAL: Comprehensive safety validation'
        try:
            # Test multiple random samples for thorough validation
            for _ in range(10):  # More thorough testing for production
                test_values = [np.random.uniform(low, high) for (low, high) in new_bounds]
                
                # Validate all critical constraints
                if test_values[1] < new_constraint_mins.get('total_air_flow_min', 150):
                    return False
                if test_values[2] < new_constraint_mins.get('wind_box_dp_min', 30):
                    return False
                if test_values[6] < new_constraint_mins.get('steam_flow_min', 120):
                    return False
                if test_values[9] < new_constraint_mins.get('coal_flow_min', 15.45):
                    return False
                if test_values[10] < new_constraint_mins.get('aph_outlet_min', 190):
                    return False
                
                # Critical air/fuel ratio validation
                if test_values[9] > 0:  # Avoid division by zero
                    air_fuel_ratio = test_values[1] / test_values[9]
                    if air_fuel_ratio < new_constraint_mins.get('air_fuel_ratio_min', 2.0):
                        return False
                
                # Additional production safety checks
                if any(v < 0 for v in test_values):  # No negative values
                    return False
                if any(np.isnan(v) or np.isinf(v) for v in test_values):  # No invalid values
                    return False
            
            return True
        except Exception as e:
            print(f"Validation error: {e}")
            return False

    def update_bounds_and_constraints_if_needed(self):
        'Update field_0001 with field_0083 field_0084 field_0085'
        if (datetime.now() - self.last_update) >= self.update_interval:
            try:
                new_bounds, new_constraint_mins = self.calculate_adaptive_bounds_and_constraints()
                
                # PRODUCTION CRITICAL: Validate before applying
                if self.validate_bounds_compatibility(new_bounds, new_constraint_mins):
                    self.current_bounds = new_bounds
                    self.current_constraint_mins = new_constraint_mins
                    self.last_update = datetime.now()
                    print(f"✅ Adaptive bounds updated at {self.last_update.strftime('%H:%M:%S')}")
                    return True
                else:
                    print(f"❌ Bounds validation FAILED at {datetime.now().strftime('%H:%M:%S')} - Safety preserved")
                    return False
            except Exception as e:
                print(f"Bounds update error: {e}")
                return False
        return False


In [ ]:
class ProductionUtkalBoilerB2:
    'Production-ready boiler optimization with adaptive bounds and safety'
    
    def __init__(self):
        # Authentication
        self.username = 'REDACTED_CONFIG_VALUE'
        self.password = 'REDACTED_CONFIG_VALUE'
        
        # MLflow setup
        os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
        os.environ['MLFLOW_TRACKING_USERNAME'] = 'REDACTED_CONFIG_VALUE'
        os.environ['MLFLOW_TRACKING_PASSWORD'] = 'REDACTED_CONFIG_VALUE'
        mlflow.set_tracking_uri('https://example.invalid/endpoint_0045')
        
        # Model features
        self.model_features = [
            'process_signal_0119',
            'process_signal_0126',
            'process_signal_0152',
            'process_signal_0146',
            'process_signal_0154',
            'process_signal_0133',
            'process_signal_0158',
            'process_signal_0128',
            'process_signal_0137',
            'process_signal_0139',
            'process_signal_0144',
            'process_signal_0121',
            'process_signal_0123',
            'process_signal_0117'
        ]
        
        # Initialize adaptive bounds manager
        self.bounds_manager = ProductionSafeAdaptiveBounds()
        
        # FIXED: Syntax error correction
        self.max_scc = 0.202  # Changed from self.max_scc <= 0.202
        
        # Load model
        self.model = self.load_model()
        
        # Update constraints
        self.update_constraints()
        
        # Initialize tags (add your PI URLs here)
        self.tags = self.initialize_tags()
        
        print('Production-ready Utkal Boiler B2 system initialized')

    def load_model(self):
        logged_model_uri = 'runs:/model_run_0002/model'
        return mlflow.xgboost.load_model(logged_model_uri)

    def update_constraints(self):
        'Update field_0005 field_0002 field_0006 field_0007 field_0009 field_0001'
        cm = self.bounds_manager.current_constraint_mins
        self.cons = [
            {'type': 'ineq', 'fun': lambda x: x[1] - cm.get('total_air_flow_min', 150)},
            {'type': 'ineq', 'fun': self.air_to_fuel_ratio},
            {'type': 'ineq', 'fun': lambda x: x[9] - cm.get('coal_flow_min', 15.45)},
            {'type': 'ineq', 'fun': lambda x: x[6] - cm.get('steam_flow_min', 120)},
            {'type': 'ineq', 'fun': lambda x: x[10] - cm.get('aph_outlet_min', 190)},
            {'type': 'ineq', 'fun': lambda x: x[2] - cm.get('wind_box_dp_min', 30)},
            {'type': 'ineq', 'fun': self.scc_constraint}
        ]

    def objective(self, x):
        input_df = pd.DataFrame([x], columns=self.model_features)
        return -self.model.predict(input_df)[0]

    def air_to_fuel_ratio(self, x):
        cm = self.bounds_manager.current_constraint_mins
        return (x[1] / x[9]) - cm.get('air_fuel_ratio_min', 2.0)

    def scc_constraint(self, x):
        return self.max_scc - (x[9] / x[6])

    def multi_start_optimization(self, live_inputs, n_runs=30):
        'Production optimization with intelligent starting points'
        current_bounds = self.bounds_manager.current_bounds
        best_solution = None
        best_efficiency = -np.inf
        successful_runs = 0
        
        for i in range(n_runs):
            try:
                # Start from current operating point (intelligent initialization)
                x0 = live_inputs.copy()
                for j in range(len(x0)):
                    if x0[j] is not None:
                        # Add small perturbation (±2% for production stability)
                        perturbation = x0[j] * 0.02 * (np.random.random() - 0.5)
                        x0[j] += perturbation
                        # Ensure within bounds
                        x0[j] = max(current_bounds[j][0], min(current_bounds[j][1], x0[j]))
                    else:
                        # Use midpoint if current input is None
                        x0[j] = (current_bounds[j][0] + current_bounds[j][1]) / 2
                
                res = minimize(
                    self.objective,
                    x0,
                    method='SLSQP',
                    bounds=current_bounds,
                    constraints=self.cons
                )
                
                if res.success and (-res.fun > best_efficiency):
                    best_solution = res.x
                    best_efficiency = -res.fun
                    successful_runs += 1
                    
            except Exception as e:
                continue
        
        print(f"Optimization: {successful_runs}/{n_runs} runs successful")
        
        if best_solution is None:
            print('⚠️ All optimization runs failed, using current inputs')
            return live_inputs, 0
            
        return best_solution, best_efficiency

    def initialize_tags(self):
        'Initialize PI tag URLs (add your actual URLs)'
        return {
            # Add your actual PI tag URLs here
            'tag1': 'url1',
            'tag2': 'url2'
            # ... etc
        }

    def safe_avg(self, values):
        'Safe average calculation'
        numeric_values = [v for v in values if isinstance(v, (int, float)) and not np.isnan(v)]
        return sum(numeric_values) / len(numeric_values) if numeric_values else None

    def read_tag_value(self, url):
        'Read single tag value with error handling'
        try:
            response = requests.get(url, auth=HTTPBasicAuth(self.username, self.password), verify=False)
            response.raise_for_status()
            data = response.json()
            value = data.get('Value')
            
            if isinstance(value, (int, float)):
                return value
            if isinstance(value, dict) and 'Value' in value:
                return value['Value']
            return None
            
        except Exception as e:
            print(f"Error reading tag: {e}")
            return None


In [ ]:
    def run_production_loop(self):
        'Production-ready main execution loop'
        print('🏭 PRODUCTION: Utkal Boiler B2 Adaptive Optimization System')
        print('=' * 60)
        
        bounds_update_count = 0
        
        while True:
            try:
                # Data collection
                stored_list = [self.read_tag_value(url) for url in self.tags.values()]
                
                # Feature engineering
                pa_fan_total = sum(stored_list[3:7]) if all(v is not None for v in stored_list[3:7]) else None
                wind_box_total = sum(stored_list[7:11]) if all(v is not None for v in stored_list[7:11]) else None
                eco_temp_avg = self.safe_avg(stored_list[14:17])
                esp_temp_avg = self.safe_avg(stored_list[18:20])
                fd_water_total = sum(stored_list[20:22]) if all(v is not None for v in stored_list[20:22]) else None
                
                model_inputs = [
                    stored_list[0], stored_list[2], pa_fan_total, wind_box_total,
                    stored_list[11], eco_temp_avg, stored_list[27], stored_list[17],
                    esp_temp_avg, fd_water_total, stored_list[22], stored_list[23],
                    stored_list[25], stored_list[26]
                ]
                
                # Add data to adaptive bounds manager
                self.bounds_manager.add_real_time_data(model_inputs)
                
                # Check for bounds update
                bounds_updated = self.bounds_manager.update_bounds_and_constraints_if_needed()
                if bounds_updated:
                    self.update_constraints()
                    bounds_update_count += 1
                    print(f"📊 Adaptive bounds update #{bounds_update_count}")
                
                # Real-time prediction and optimization
                scc_prediction = self.model.predict([model_inputs])[0]
                actual_scc = stored_list[24] if stored_list[24] is not None else scc_prediction
                
                # Conservative prediction adjustment
                if scc_prediction > actual_scc:
                    scc_prediction = actual_scc
                
                # Run optimization
                x_opt, eff_opt = self.multi_start_optimization(model_inputs, n_runs=30)
                optimized_scc = x_opt[9] / x_opt[6] if x_opt[6] > 0 else scc_prediction
                
                # Production monitoring output
                print(f"⏰ Time: {datetime.now().strftime('%H:%M:%S')}")
                print(f"📈 Current SCC: {actual_scc:.4f}")
                print(f"🔮 Predicted SCC: {scc_prediction:.4f}")
                print(f"⚡ Optimized SCC: {optimized_scc:.4f}")
                print(f"🎯 Efficiency gain: {eff_opt:.4f}")
                
                # Production safety check
                if optimized_scc > self.max_scc:
                    print(f"⚠️ WARNING: Optimized SCC {optimized_scc:.4f} exceeds limit {self.max_scc}")
                
                next_update = 60 - (datetime.now() - self.bounds_manager.last_update).total_seconds()
                print(f"⏳ Next bounds update in: {next_update:.0f}s")
                print('-' * 60)
                
                # Production sleep interval
                time.sleep(10)
                
            except Exception as e:
                print(f"❌ Production error: {e}")
                time.sleep(10)

# Add the method to the class
ProductionUtkalBoilerB2.run_production_loop = run_production_loop


In [ ]:
if __name__ == '__main__':
    print('🚀 PRODUCTION DEPLOYMENT: Enhanced Adaptive Boiler Optimization')
    print('🔒 Safety Features: Emergency limits, validation, conservative learning')
    print('=' * 70)
    
    # Initialize production system
    production_system = ProductionUtkalBoilerB2()
    
    # Run production loop
    production_system.run_production_loop()
